# 03 · QLoRA's NF4 and double quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/03_nf4_quantization.ipynb) &nbsp; 📖 Concept deck: [LoRA and QLoRA](https://nilesh-salpe.github.io/visual-ai/post-training/03-lora-qlora/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** QLoRA's 4-bit NormalFloat (NF4) blockwise quantizer, a comparison with uniform INT4 on Gaussian weights, and the double-quantization memory arithmetic.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

# Exact NF4 values from the QLoRA paper (Appendix E)
NF4 = np.array([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                0.7229568362236023, 1.0])
INT4 = np.linspace(-1, 1, 16)   # 16 evenly spaced levels for comparison

## Task 1: Blockwise absmax quantization to a codebook

Split the flattened weights into blocks of `block` values. For each block:
1. the scale is $c = \max |w|$ in the block (the "absmax" quantization constant);
2. normalize $w/c \in [-1, 1]$;
3. store the **index** of the nearest codebook value (4 bits for 16 values).

Return `(idx, scales)`, where `idx` has shape [n_blocks, block] and `scales` [n_blocks]. Assume `w.size` is a multiple of `block`.

In [ ]:
def quantize_blockwise(w, codebook, block=64):
    # TODO
    raise NotImplementedError

def dequantize_blockwise(idx, scales, codebook, shape):
    # TODO: codebook[idx] * scales[:, None], reshaped
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_quantize_blockwise(f):
    deq = dequantize_blockwise_solution if f.__name__.endswith('_solution') else dequantize_blockwise
    w = np.random.default_rng(1).normal(size=(8, 32))
    idx, sc = f(w, NF4, 64)
    assert idx.shape == (4, 64) and sc.shape == (4,), "4 blocks of 64"
    assert np.allclose(sc, np.abs(w.reshape(4, 64)).max(1)), "scale = absmax per block"
    assert idx.min() >= 0 and idx.max() <= 15
    wq = deq(idx, sc, NF4, w.shape)
    assert wq.shape == w.shape
    assert np.all(np.abs(wq) <= np.abs(w).max() + 1e-12)
    nearest = NF4[np.abs((w.reshape(4, 64) / sc[:, None])[..., None] - NF4).argmin(-1)]
    assert np.allclose(wq.reshape(4, 64), nearest * sc[:, None]), "each value must map to the NEAREST codebook entry"

check("quantize_blockwise", _test_quantize_blockwise, quantize_blockwise)

## Task 2: Double quantization: the cost of the scales

Each block stores one scale. With FP32 scales and blocksize 64 that's $32/64 = 0.5$ extra bits per parameter.
**Double quantization** quantizes the scales themselves to 8 bits, in blocks of 256 with one FP32 constant each:
$$\frac{8}{64} + \frac{32}{64\cdot 256} = 0.127 \text{ bits per parameter}$$
QLoRA: a reduction of 0.373 bits per parameter, about 3 GB for a 65B model. Return the overhead in bits per parameter.

In [ ]:
def quant_overhead_bits(block=64, double=False, block2=256):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_quant_overhead_bits(f):
    assert np.isclose(f(64), 0.5)
    assert np.isclose(f(64, True), 0.126953125), f"got {f(64, True)} (the paper rounds this to 0.127)"
    saved_gb = (f(64) - f(64, True)) * 65e9 / 8 / 1e9
    assert 2.9 < saved_gb < 3.1, f"saving for 65B should be ~3 GB, got {saved_gb:.2f}"

check("quant_overhead_bits", _test_quant_overhead_bits, quant_overhead_bits)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Blockwise absmax quantization to a codebook
def quantize_blockwise_solution(w, codebook, block=64):
    b = w.reshape(-1, block)
    scales = np.abs(b).max(1)
    idx = np.abs((b / scales[:, None])[..., None] - codebook).argmin(-1)
    return idx, scales

def dequantize_blockwise_solution(idx, scales, codebook, shape):
    return (codebook[idx] * scales[:, None]).reshape(shape)

check("quantize_blockwise (reference)", _test_quantize_blockwise, quantize_blockwise_solution)

In [ ]:
# @title Solution for Task 2: Double quantization: the cost of the scales
def quant_overhead_bits_solution(block=64, double=False, block2=256):
    return 8 / block + 32 / (block * block2) if double else 32 / block

check("quant_overhead_bits (reference)", _test_quant_overhead_bits, quant_overhead_bits_solution)

---
## Experiments
**Why "NormalFloat"?** Pretrained weights are roughly zero-centred normal. NF4 puts its 16 levels at quantiles of a normal distribution, dense near 0 where most weights are, so it wastes fewer levels than evenly spaced INT4. QLoRA calls NF4 "information-theoretically optimal for normally distributed weights". Measure both on Gaussian weights, then add up the memory for a 65B model.

In [ ]:
w = np.random.default_rng(0).normal(0, 0.02, size=(4096, 1024))
for name, cb in [('INT4 (uniform)', INT4), ('NF4', NF4)]:
    idx, sc = quantize_blockwise_solution(w, cb)
    wq = dequantize_blockwise_solution(idx, sc, cb, w.shape)
    print(f"{name:15} relative RMSE {np.sqrt(((w - wq) ** 2).mean()) / w.std():.4f}   levels used: {len(np.unique(idx))}/16")
P = 65e9
print(f"\n65B weights: BF16 {P * 2 / 1e9:.0f} GB  ->  NF4 {P * 0.5 / 1e9:.1f} GB + scales {P * quant_overhead_bits_solution(64) / 8 / 1e9:.1f} GB"
      f" (double-quantized: {P * quant_overhead_bits_solution(64, True) / 8 / 1e9:.1f} GB)")
print("QLoRA reports fine-tuning 65B on a single 48 GB GPU (vs >780 GB for 16-bit full fine-tuning).")

---
## Takeaways
* QLoRA = a frozen base model stored in 4-bit NF4, dequantized to BF16 on the fly for each matmul, plus LoRA adapters trained in 16-bit.
* NF4 spends its 16 levels where normally distributed weights actually are, so it has lower error than uniform INT4.
* Double quantization shrinks the scale overhead from 0.5 to 0.127 bits per parameter. Paged optimizers absorb memory spikes. Together they let a 65B model fine-tune on one 48 GB GPU.